In [4]:
import numpy as np

In [6]:
matrix = np.array([[2,5,7],
                  [6,0,9],
                  [0,0,1]])

# A slice of 0:2 translates to [0,2), which means 0 and 1 only

In [7]:
sub_matrix = matrix[0:2, 0:2]

In [8]:
print(sub_matrix)

[[2 5]
 [6 0]]


In [9]:
sub_matrix[0,1] = 99

In [10]:
print(sub_matrix)
print(matrix)

[[ 2 99]
 [ 6  0]]
[[ 2 99  7]
 [ 6  0  9]
 [ 0  0  1]]


# This means that Sliced matrix is just a refernece to the original matrix

In [11]:
# Transposed Matrix
t_matrix = matrix.T

In [12]:
print(t_matrix)

[[ 2  6  0]
 [99  0  0]
 [ 7  9  1]]


In [16]:
print(matrix.strides)
print(t_matrix.strides)
# moment of realisation

(24, 8)
(8, 24)


### 🔄 The Transpose "Magic": Swapping Strides, Not Data

When you transpose a NumPy array, you want rows to become columns and columns to become rows. 
**Crucially, NumPy changes *zero* data in your RAM.** 

Instead, it literally just **swaps the stride values** in the array metadata header. This is a constant-time operation ($O(1)$), regardless of matrix size.

#### 🔍 Let's Prove It with Your 3x3 Matrix

Assuming `int64` (8 bytes per element), your original matrix had strides of `(24, 8)`:
*   **Column Stride (8):** To move to the next column, jump **8 bytes**.
*   **Row Stride (24):** To move to the next row, skip the entire row (3 elements × 8 bytes = **24 bytes**).

When you call `.transpose()`, NumPy swaps these values to `(8, 24)`:

1.  **Next Column (`t_matrix[0, 1]`)**:
    *   NumPy looks at the *new* column stride: **24 bytes**.
    *   It jumps 24 bytes ahead in the original memory block.
    *   **Result:** It lands exactly on the element that used to be in the *next row* of the original matrix.

2.  **Next Row (`t_matrix[1, 0]`)**:
    *   NumPy looks at the *new* row stride: **8 bytes**.
    *   It jumps just 8 bytes ahead.
    *   **Result:** It lands on the element that used to be the *adjacent column* in the original matrix.

#### 💡 Why This is a Systems Masterpiece

| Operation | C++ Approach (Physical Swap) | NumPy Approach (Strides) |
| :--- | :--- | :--- |
| **Time Complexity** | $O(N^2)$ (Must move every byte) | $O(1)$ (Just swap two numbers) |
| **Memory Access** | Massive cache misses | Zero

In [17]:
arr = np.array([2,1,9,0,7])

In [20]:
result = arr + 2 # add 2 to every single element

In [19]:
print(result)

[ 4  3 11  2  9]


Under the hood, NumPy does not execute a Python loop. Because the data type is homogeneous and packed contiguously, NumPy passes the entire memory block directly to compiled C/C++ loops. Even better, if your CPU supports it, it leverages SIMD (Single Instruction, Multiple Data) instructions.

In [28]:
arr2 = np.array([4,0,9,1,4])

In [32]:
print("Addition : ", arr + arr2) # Adds correspoding elements (Shape should be same)
print("Multiplication : ", arr * arr2) # Multiplies corresponding elements (Not dot product) (Shape should be same)

Addition :  [ 6  1 18  1 11]
Multiplication :  [ 8  0 81  0 28]


In [33]:
img = np.array([[4,5,6],
              [5,1,0],
              [4,9,8],
              [1,4,7]])

In [34]:
sub = np.array([0.5, 0.1, 0.2])

In [35]:
print(img - sub)

[[ 3.5  4.9  5.8]
 [ 4.5  0.9 -0.2]
 [ 3.5  8.9  7.8]
 [ 0.5  3.9  6.8]]


In [36]:
sub2 = np.array([0.5, 0.1, 0.2, 0.1])

In [37]:
print(img - sub2)

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 

In [43]:
a = np.array([[3,4,1],
             [9,9,0]]) # 2x3

In [44]:
b = np.array([[3,4],
             [9,9],
             [6,1]]) #3x2

In [48]:
c = a @ b

In [49]:
print("Dot product : \n", c)

Dot product : 
 [[ 51  49]
 [108 117]]


In [50]:
print(c.shape)

(2, 2)


When you execute A @ B, NumPy doesn't run a naive loop. Under the hood, it hands the raw memory pointers over to highly optimized linear algebra subroutines written in C/Fortran called BLAS (Basic Linear Algebra Subprograms)—usually OpenBLAS, Intel MKL, or Apple Performance Libraries depending on your machine.

These libraries use complex algorithmic strategies like Cache Blocking (breaking matrices down into tiny sub-matrices that fit entirely inside your CPU's L1/L2 cache lines) and explicit multi-threaded SIMD parallelization to compute the dot product thousands of times faster than standard code.

In [51]:
x = np.random.randn(1000,1000)

In [55]:
%timeit x @ x

37.3 ms ± 351 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [59]:
x_t = x.T

In [60]:
%timeit x_t @ x_t

37.6 ms ± 163 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


When you execute `A @ B` in NumPy, it bypasses naive loops by delegating to highly optimized **BLAS** (Basic Linear Algebra Subprograms) libraries like OpenBLAS, Intel MKL, or Apple Performance Libraries. These C/Fortran backends achieve massive speedups through **Cache Blocking** (fitting sub-matrices into CPU cache) and **SIMD parallelization**.

Modern BLAS implementations are resilient to memory layout changes. When encountering non-contiguous matrices (e.g., `X_T @ X_T`), the backend analyzes memory strides and applies one of two optimizations:

1. **Internal Transpose ("Lazy Swap")**: The library leverages the natural iteration of rows and columns in multiplication algorithms. Feeding it transposed matrices can align memory access perfectly with its internal loop structures, avoiding data movement.
2. **Dynamic Copying**: If strides create a worst-case cache scenario, the backend allocates a temporary buffer to copy data into a contiguous block. This overhead is negligible compared to the performance gains from hardware-speed SIMD execution on the contiguous data.

# Conditional Manuplation and Boolean Algebra

In [80]:
scores = np.array([30,92,19,30,81,18,46])

In [81]:
mask = scores <= 30

In [82]:
print(mask)

[ True False  True  True False  True False]


In [83]:
Failing_scores = scores[mask]

In [84]:
print(Failing_scores)

[30 19 30 18]


In [92]:
cond_mask = (scores >= 20) & (scores <= 80) # use the parentheses around each condition, or Python's operator precedence will break.

In [93]:
print(scores[cond_mask])

[30 30 46]


In [94]:
Failing_scores[1] = 99

In [95]:
scores

array([30, 92, 19, 30, 81, 18, 46])

In [96]:
Failing_scores

array([30, 99, 30, 18])